# scDecontX in five minutes

`scdecontx` removes ambient RNA contamination from an `AnnData` object
with a single function call.

This notebook makes a small toy dataset (so it runs anywhere), removes
the contamination, and checks the result. Run the cells in order with
Shift+Enter.

In [1]:
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

import scdecontx as dx

## A toy dataset

We simulate 1,500 cells in 5 clusters. Each cell gets 3,000 counts:
most from its own cluster's genes, plus ~20% of "ambient" RNA leaked
from the other clusters - the contamination we want to remove.

In [2]:
rng = np.random.default_rng(0)  # fixed seed, so results are reproducible

n_cells, n_genes, n_clusters = 1500, 2000, 5

# Each cluster has its own gene expression profile (each row sums to 1).
profiles = rng.gamma(0.3, 1.0, size=(n_clusters, n_genes))
profiles = profiles / profiles.sum(axis=1, keepdims=True)

# The ambient profile is the average across clusters.
ambient = profiles.mean(axis=0)

# Assign each cell to a cluster and give it a contamination fraction (~20%).
cluster = rng.integers(n_clusters, size=n_cells)
true_contamination = rng.beta(2, 8, size=n_cells)

# Each cell: counts from its own cluster's profile plus ambient counts.
counts = np.zeros((n_cells, n_genes), dtype=np.int64)
for i in range(n_cells):
    n_ambient = int(3000 * true_contamination[i])
    n_native = 3000 - n_ambient
    counts[i] = rng.multinomial(n_native, profiles[cluster[i]])
    counts[i] += rng.multinomial(n_ambient, ambient)

Now we put the counts into an `AnnData` object, the standard container
for single-cell data. `obs` holds per-cell information: the cluster and
the true contamination (which real data would not have - we keep it only
to check the answer).

In [3]:
adata = ad.AnnData(
    X=sp.csr_matrix(counts),
    obs=pd.DataFrame(
        {
            "cluster": pd.Categorical(cluster.astype(str)),
            "true_contamination": true_contamination,
        },
        index=[f"cell{i}" for i in range(n_cells)],
    ),
)
adata

AnnData object with n_obs × n_vars = 1500 × 2000
    obs: 'cluster', 'true_contamination'
    layers: None (.X)

## Run scDecontX

`dx.run` needs two things:

- raw counts in `adata.X` (as here), and
- a cluster label for each cell in `adata.obs` - ours is `"cluster"`.

In [4]:
dx.run(adata, groupby="cluster")

It writes the results back into `adata`:

- `obs["scdecontx_contamination"]` - estimated contamination per cell
- `layers["scdecontx_counts"]` - the decontaminated counts
- `uns["scdecontx"]` - details of the fit

## Check the result

Because we made this dataset, we know each cell's true contamination,
so we can measure the error of the estimates.

In [5]:
est = adata.obs["scdecontx_contamination"]
true = adata.obs["true_contamination"]

print("mean true contamination:      ", round(true.mean(), 3))
print("mean estimated contamination: ", round(est.mean(), 3))
print("mean absolute error:          ", round(np.abs(est - true).mean(), 3))

mean true contamination:       0.204
mean estimated contamination:  0.17
mean absolute error:           0.035


A peek at the decontaminated counts (first 3 cells, first 5 genes):

In [6]:
adata.layers["scdecontx_counts"][:3, :5].toarray()

array([[0.02153088, 0.97895855, 0.33269822, 0.        , 0.17196711],
       [0.        , 0.        , 0.        , 0.59919723, 0.        ],
       [0.        , 0.        , 1.85283755, 0.        , 0.        ]])

## Next

Use the decontaminated counts for the rest of your analysis:

```python
adata.X = adata.layers["scdecontx_counts"]
```

Options you may need on real data: `batch_key=` fits each sample
separately, `layer=` reads counts from a layer instead of `.X`,
`background=` uses raw empty droplets as the contamination profile,
and `copy=True` returns a new object instead of writing to `adata`.

For the full parameter list see the README table; for method details
see `docs/methods.md`.